<a href="https://colab.research.google.com/github/ankith-30403/ML-Practical-Lab-Experiments/blob/main/Week8_practical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import AdaBoostRegressor # Changed from AdaBoostClassifier
from sklearn.tree import DecisionTreeRegressor # Changed from DecisionTreeClassifier
from sklearn.metrics import mean_squared_error, r2_score # Changed metrics for regression
from xgboost import XGBRegressor # Changed from XGBClassifier

RANDOM_STATE = 42
TARGET_COL = "median_house_value" # Updated target column for California Housing
DATA_PATH = "/content/sample_data/california_housing_train.csv" # Updated data path


# --------------------------------------------------------------------------
# 1. Load + preprocess (plain top-level script code, no function wrapper)
# --------------------------------------------------------------------------
df = pd.read_csv(DATA_PATH)

# Drop the injected-anomaly flag; it's a data-quality marker, not a real feature
# (No 'IsAnomaly' column in California Housing dataset)
# if "IsAnomaly" in df.columns:
#     df = df.drop(columns=["IsAnomaly"])

y = df[TARGET_COL] # Removed .astype(int) as it's a regression target
X = df.drop(columns=[TARGET_COL])

cat_cols = X.select_dtypes(exclude="number").columns.tolist()
num_cols = X.select_dtypes(include="number").columns.tolist()

# Label-encode categoricals (fine for tree-based boosters; no dummy blow-up)
# California Housing is primarily numerical, so this might not be needed.
encoders = {}
if cat_cols:
    for c in cat_cols:
        le = LabelEncoder()
        X[c] = le.fit_transform(X[c].astype(str))
        encoders[c] = le

# Impute missing numerics (median) — AdaBoost's DecisionTree base estimator
# can't handle NaN natively, unlike XGBoost, so both models get the same clean input
imputer = SimpleImputer(strategy="median")
X[num_cols] = imputer.fit_transform(X[num_cols])

# Scale numerics — helps AdaBoost's (default) shallow-tree base estimator converge cleanly
scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])


# --------------------------------------------------------------------------
# 2. Train / validation / test split (also plain top-level code)
# --------------------------------------------------------------------------
VAL_SIZE = 0.15
TEST_SIZE = 0.15

# First carve off test, then split remainder into train/val
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE # Removed stratify due to regression target
)
val_ratio = VAL_SIZE / (1 - TEST_SIZE)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=val_ratio,
    random_state=RANDOM_STATE # Removed stratify due to regression target
)

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")


# --------------------------------------------------------------------------
# 3. Benchmark function (kept as a function, since it's meant to be reusable)
# --------------------------------------------------------------------------
def boosting_benchmark(X_train, y_train, X_val, y_val):
    """
    Fits AdaBoost (adaptive sample-weight boosting) and XGBoost
    (gradient boosting with early stopping on the validation set),
    evaluates both on the validation set, and returns a DataFrame
    of results sorted by val_accuracy descending.
    """
    results = []

    # ---- Model 1: AdaBoost ------------------------------------------------
    # AdaBoost re-weights training samples each round: misclassified samples
    # get higher weight so the next weak learner focuses on them.
    ada_base = DecisionTreeRegressor(max_depth=4, random_state=RANDOM_STATE) # Changed to Regressor, increased max_depth
    ada = AdaBoostRegressor( # Changed to Regressor
        estimator=ada_base,
        n_estimators=200,
        learning_rate=0.5,
        random_state=RANDOM_STATE,
    )

    t0 = time.time()
    ada.fit(X_train, y_train)
    ada_fit_time = time.time() - t0

    ada_val_pred = ada.predict(X_val)

    results.append({
        "model": "AdaBoost",
        "val_mse": mean_squared_error(y_val, ada_val_pred), # Changed to Mean Squared Error
        "val_rmse": np.sqrt(mean_squared_error(y_val, ada_val_pred)), # Added RMSE
        "val_r2": r2_score(y_val, ada_val_pred), # Added R-squared
        "best_n_estimators": ada.n_estimators,
        "fit_time_sec": round(ada_fit_time, 2),
    })

    # ---- Model 2: XGBoost with early stopping -----------------------------
    # early_stopping_rounds lives on the constructor in xgboost>=2.0; fit()
    # is given eval_set and stops once val logloss hasn't improved in N rounds.
    xgb = XGBRegressor( # Changed to Regressor
        n_estimators=1000,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="rmse", # Already set to rmse, appropriate for regression
        early_stopping_rounds=30,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    t0 = time.time()
    xgb.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )
    xgb_fit_time = time.time() - t0

    xgb_val_pred = xgb.predict(X_val)

    results.append({
        "model": "XGBoost",
        "val_mse": mean_squared_error(y_val, xgb_val_pred), # Changed to Mean Squared Error
        "val_rmse": np.sqrt(mean_squared_error(y_val, xgb_val_pred)), # Added RMSE
        "val_r2": r2_score(y_val, xgb_val_pred), # Added R-squared
        "best_n_estimators": xgb.best_iteration + 1,
        "fit_time_sec": round(xgb_fit_time, 2),
    })

    # For a regression problem, we typically sort by a metric like RMSE or R-squared.
    # Sorting by RMSE (lower is better)
    results_df = pd.DataFrame(results).sort_values(
        "val_rmse", ascending=True
    ).reset_index(drop=True)

    return results_df


# --------------------------------------------------------------------------
# 4. Run
# --------------------------------------------------------------------------
# Note: The boosting_benchmark function and its metrics are currently set up
# for classification. Running it directly with a regression target will cause errors
# or produce meaningless results for accuracy/f1/roc_auc.
# If you intend to proceed with regression, the `boosting_benchmark` function
# would need to be re-written to use regression models (e.g., AdaBoostRegressor,
# XGBRegressor) and appropriate regression metrics (e.g., Mean Squared Error).

# Placeholder to avoid error if `leaderboard` is used later without changes.
# leaderboard = pd.DataFrame(columns=["model", "fit_time_sec"])
# print("Data loaded and preprocessed for regression. Model benchmarking needs adaptation for regression.")

leaderboard = boosting_benchmark(X_train, y_train, X_val, y_val) # Uncommented
print("\nValidation leaderboard (sorted by val_rmse):") # Updated print statement
print(leaderboard.to_string(index=False))

leaderboard.to_csv("boosting_benchmark_results.csv", index=False)
print("\nSaved results to boosting_benchmark_results.csv")


Train: (11900, 8) | Val: (2550, 8) | Test: (2550, 8)

Validation leaderboard (sorted by val_rmse):
   model      val_mse     val_rmse   val_r2  best_n_estimators  fit_time_sec
 XGBoost 2.334622e+09 48317.929084 0.825750                954          1.53
AdaBoost 7.106037e+09 84297.315047 0.469623                200          1.90

Saved results to boosting_benchmark_results.csv
